# Proyecto de Clasificación con dataset HIGGS


In [12]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

data_path = "../../../src/data/HIGGS/"

# 1. Carga de datos
print("Cargando los datos...")
train = pd.read_csv(data_path + "train_200000.csv", header=None)
validation = pd.read_csv(data_path + "validation.csv", header=None)
validation_manifest = pd.read_csv(data_path + "validation.manifest.csv")

# Separar la primera columna como variable objetivo y el resto como características
y_train = train.iloc[:, 0]
X_train = train.iloc[:, 1:]

y_val = validation.iloc[:, 0]
X_val = validation.iloc[:, 1:]

print("Dimensiones de entrenamiento:", X_train.shape, y_train.shape)
print("Dimensiones de validación:", X_val.shape, y_val.shape)


Cargando los datos...
Dimensiones de entrenamiento: (200000, 28) (200000,)
Dimensiones de validación: (500000, 28) (500000,)


In [13]:
# 2. Preprocesamiento
print("\nAplicando StandardScaler...")
scaler = StandardScaler()

# El ajuste (fit) se hace únicamente con train
scaler.fit(X_train)

# Utilizar transform() tanto en set de entrenamiento como en validation
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
print("Preprocesamiento terminado.")



Aplicando StandardScaler...
Preprocesamiento terminado.


In [14]:
# 3. Entrenar modelos y 4. Evaluar
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "SGD Classifier": SGDClassifier(max_iter=1000, random_state=42)
}

resultados = []
mejor_modelo = None
mejor_f1 = 0
nombre_mejor = ""

for nombre, modelo in modelos.items():
    print(f"\nEntrenando {nombre}...")
    start_time = time.time()
    modelo.fit(X_train_scaled, y_train)
    tiempo_entrenamiento = time.time() - start_time
    
    # Predecir sobre el validación ya escalado
    y_pred = modelo.predict(X_val_scaled)
    
    acc = accuracy_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    resultados.append({
        "Modelo": nombre,
        "Accuracy": acc,
        "Recall": rec,
        "F1": f1,
        "Training time (s)": tiempo_entrenamiento
    })
    
    if f1 > mejor_f1:
        mejor_f1 = f1
        mejor_modelo = modelo
        nombre_mejor = nombre

# Comparar resultados
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

print(f"\nEl modelo ganador es: {nombre_mejor} con un F1 de {mejor_f1:.4f}")



Entrenando Logistic Regression...

Entrenando Decision Tree...

Entrenando SGD Classifier...


,Modelo,Accuracy,Recall,F1,Training time (s)
0,Logistic Regression,0.641516,0.740605,0.686495,0.559573
1,Decision Tree,0.694144,0.727529,0.716007,6.822306
2,SGD Classifier,0.640100,0.783227,0.697579,1.765170



El modelo ganador es: Decision Tree con un F1 de 0.7160


In [16]:
# 5. Generar archivo de predicciones
print(f"Generando predicciones usando {nombre_mejor}...")

# Predicciones en validación
predicciones = mejor_modelo.predict(X_val_scaled)

# Juntar IDs con predicciones usando validation_manifest
df_final = pd.DataFrame({
    'observation_id': validation_manifest['observation_id'],
    'prediction': predicciones.astype(int)
})

# Exportar a CSV
output_file = "validation_predictions_HIGGS.csv"
df_final.to_csv(output_file, index=False)
print(f"Predicciones guardadas exitosamente en {output_file}")

# Mostrar algunas filas del archivo final
df_final.head()


Generando predicciones usando Decision Tree...
Predicciones guardadas exitosamente en validation_predictions_HIGGS.csv


,observation_id,prediction
0,HIGGS.csv.gz:1,1
1,HIGGS.csv.gz:18,0
2,HIGGS.csv.gz:22,1
3,HIGGS.csv.gz:28,0
4,HIGGS.csv.gz:39,0
